# OpsPilot — Phase 4: Embeddings & Semantic Retrieval (RAG)
**Capstone | Scenario 1: IT Operations Copilot | Track A: LangChain**

This notebook:
1. Prepares and chunks documents for embedding
2. Builds a ChromaDB vector store using OpenAI embeddings
3. Implements semantic retrieval
4. Compares responses **with vs without RAG**
5. Tests relevance and handles missing-data cases

> ⚠️ **Run cells top to bottom. Do not skip cells.**

---

## Cell 1 — Install Dependencies

In [ ]:
%pip install openai pandas chromadb pysqlite3-binary --quiet
print('✅ Dependencies installed')

## Cell 2 — All Imports

In [ ]:
# ── SQLite3 patch: required for ChromaDB on Vocareum (sqlite < 3.35.0) ────────
__import__("pysqlite3")
import sys
sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

import os, re, json, time, logging
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

os.makedirs('logs', exist_ok=True)
os.makedirs('data/vectorstore', exist_ok=True)

print('✅ All imports loaded')
print(f'   chromadb : {chromadb.__version__}')
print(f'   pandas   : {pd.__version__}')

## Cell 3 — API Key & Data Load

In [ ]:
# Set API key (Vocareum environment variable is already set)
os.environ['OPENAI_MODEL'] = 'gpt-4o-mini'
client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
MODEL  = os.environ['OPENAI_MODEL']

# Load structured data
incidents = pd.read_csv('data/incidents.csv')
incidents['opened_at']    = pd.to_datetime(incidents['opened_at'],    errors='coerce')
incidents['resolved_at']  = pd.to_datetime(incidents['resolved_at'],  errors='coerce')
incidents['mttr_minutes'] = pd.to_numeric(incidents['mttr_minutes'],  errors='coerce')
services    = pd.read_csv('data/services.csv')
sla_targets = pd.read_csv('data/sla_targets.csv')

print(f'✅ {len(incidents)} incidents | {len(services)} services loaded')
print(f'✅ OpenAI client ready ({MODEL})')

## Cell 4 — Document Preparation & Chunking
Three document types are prepared:
1. **Runbooks** — procedural knowledge (escalation paths, known issues)
2. **Service summaries** — per-service incident statistics
3. **SLA definitions** — target MTTR values

In [ ]:
def chunk_text(text, chunk_size=300, overlap=60):
    """Split text into overlapping word-count chunks."""
    words  = text.split()
    chunks = []
    start  = 0
    while start < len(words):
        chunks.append(' '.join(words[start:start+chunk_size]))
        start += chunk_size - overlap
    return chunks

docs = []

# 1. Runbooks
runbook_dir = Path('data/runbooks')
for rb_file in runbook_dir.glob('*.txt'):
    text = rb_file.read_text()
    for i, chunk in enumerate(chunk_text(text, 300, 60)):
        docs.append({
            'id':       f'rb_{rb_file.stem}_{i}',
            'text':     chunk,
            'metadata': {'source':'runbook','filename':rb_file.name,'service':rb_file.stem,'chunk':i}
        })
    print(f'  📄 {rb_file.name} → {len(chunk_text(text,300,60))} chunks')

# 2. Per-service incident summaries
for _, srow in services.iterrows():
    svc      = srow['service_name']
    svc_inc  = incidents[incidents['service'] == svc]
    if svc_inc.empty: continue
    top3     = svc_inc['root_cause'].value_counts().head(3)
    breaches = (svc_inc['sla_breached'] == 'Yes').sum()
    opens    = svc_inc['status'].isin(['Open','In Progress']).sum()
    avg_mttr = svc_inc['mttr_minutes'].mean()
    summary  = (f'Service summary for {svc}:\n'
                f'Total incidents: {len(svc_inc)} | Open: {opens} | '
                f'SLA breaches: {breaches} ({breaches/max(len(svc_inc),1)*100:.1f}%)\n'
                f'Average MTTR: {avg_mttr:.0f} min | Criticality: {srow["criticality"]} | '
                f'Uptime 30d: {srow["uptime_pct_30d"]}%\n'
                f'Top root causes:\n' +
                '\n'.join(f'  - {c}: {n} incidents' for c,n in top3.items()))
    docs.append({
        'id':       f'svc_{svc.replace("-","_")}',
        'text':     summary,
        'metadata': {'source':'service_summary','service':svc,'filename':'','chunk':0}
    })

print(f'  📊 {len(services)} service summaries prepared')

# 3. SLA definitions
sla_text = ('NovaTech SLA Definitions:\n'
            'P1 Critical: full outage. MTTR target 60 min.\n'
            'P2 High: major degradation. MTTR target 240 min (4 hours).\n'
            'P3 Medium: minor impact. MTTR target 1440 min (24 hours).\n'
            'P4 Low: cosmetic. MTTR target 4320 min (72 hours).\n'
            'Escalation: if MTTR will exceed SLA, escalate immediately.\n'
            'P1 incidents require Ops Lead notification within 10 minutes.')
docs.append({'id':'sla_definitions','text':sla_text,
             'metadata':{'source':'sla','service':'all','filename':'','chunk':0}})

print(f'\n✅ Total chunks prepared: {len(docs)}')
print('\nSample chunk (auth-service runbook):')
sample = next(d for d in docs if 'rb_auth' in d['id'])
print(f'  ID: {sample["id"]}')
print(f'  Text: {sample["text"][:200]}...')

## Cell 5 — Build ChromaDB Vector Store
Embeds all chunks using `text-embedding-3-small` and persists to disk.

In [ ]:
embed_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key    = os.environ['OPENAI_API_KEY'],
    model_name = 'text-embedding-3-small',
)

chroma_client = chromadb.PersistentClient(path='data/vectorstore')

# Drop and recreate for a clean build
try:
    chroma_client.delete_collection('ops_knowledge')
    print('  Cleared existing collection')
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name='ops_knowledge',
    embedding_function=embed_fn,
    metadata={'hnsw:space': 'cosine'},
)

print(f'Embedding {len(docs)} chunks (may take ~15 seconds)...')
t0 = time.time()

BATCH = 50
for i in range(0, len(docs), BATCH):
    batch = docs[i:i+BATCH]
    collection.upsert(
        ids       = [d['id']       for d in batch],
        documents = [d['text']     for d in batch],
        metadatas = [d['metadata'] for d in batch],
    )

elapsed = round(time.time() - t0, 1)
print(f'✅ Vector store built in {elapsed}s')
print(f'   Total chunks indexed: {collection.count()}')
print(f'   Storage: data/vectorstore/')

## Cell 6 — Retrieval Function & Relevance Test

In [ ]:
def retrieve(query, top_k=3, min_similarity=0.25):
    """Embed query → find top_k similar chunks → filter by min_similarity."""
    results = collection.query(
        query_texts=[query],
        n_results=top_k,
        include=['documents','metadatas','distances'],
    )
    hits = []
    for doc, meta, dist in zip(results['documents'][0],
                                results['metadatas'][0],
                                results['distances'][0]):
        sim = round(1 - dist/2, 3)   # cosine distance → similarity score
        if sim >= min_similarity:
            hits.append({
                'text':       doc,
                'source':     meta.get('source',''),
                'service':    meta.get('service',''),
                'filename':   meta.get('filename',''),
                'similarity': sim,
            })
    return hits

def format_kb_context(hits):
    if not hits:
        return 'No relevant knowledge base entries found.'
    parts = ['Retrieved knowledge base context:']
    for i, h in enumerate(hits, 1):
        src = h['filename'] if h['filename'] else h['source']
        parts.append(f'\n[KB-{i}] Source: {src} (relevance: {h["similarity"]:.2f})\n{h["text"]}')
    return '\n'.join(parts)

# Relevance test — 3 queries
RETRIEVAL_TESTS = [
    ('auth-service escalation path',          'What is the escalation path for auth-service P1?'),
    ('payments-api known issues',             'What are known issues with payments-api?'),
    ('unrelated — should return low scores',  'What is the weather today in Mumbai?'),
]

print('RETRIEVAL RELEVANCE TEST')
print('='*65)
for label, q in RETRIEVAL_TESTS:
    hits = retrieve(q, top_k=3)
    print(f'\nQuery : {q}')
    print(f'Label : {label}')
    if not hits:
        print('  → No results above similarity threshold (expected for unrelated query)')
    for h in hits:
        src = h['filename'] if h['filename'] else h['source']
        print(f'  [{h["similarity"]:.3f}] {src}: {h["text"][:100]}...')
print('\n✅ Retrieval function ready')

## Cell 7 — RAG Prompt & LLM Functions

In [ ]:
# Re-configure logger
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
logging.basicConfig(
    filename='logs/rag_interactions.log', level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

def _strip_pii(text):
    text = re.sub(r'\bANL-\d{3}\b', '[ANALYST]', text)
    text = re.sub(r'\b[A-Z][a-z]+ [A-Z][a-z]+\b', '[NAME]', text)
    return text

RAG_PROMPT = """You are OpsPilot — a read-only AI Decision Support Copilot for NovaTech's IT Operations team.

SCOPE: IT incidents, SLA tracking, MTTR, root causes, operational trends, runbook procedures.
NOT in scope: hiring, budgets, HR, vendor decisions.

SAFETY: Never restart/deploy/trigger anything. Never fabricate data. Escalate ambiguous cases.

YOU HAVE TWO SOURCES:
A) Structured data — live incident/service metrics
B) Knowledge base — runbooks, service summaries, SLA definitions
Use A for numbers/counts. Use B for procedures and escalation paths.
Cite sources: [from data] or [from runbook] or [from KB].
If KB relevance < 0.4, note it may not be directly applicable.

FORMAT: Direct answer first. Bullet points. ⚠️ for uncertainty. End with Recommend: if complex.

--- STRUCTURED DATA ---
{structured_context}

--- KNOWLEDGE BASE ---
{kb_context}"""

WITHOUT_RAG_PROMPT = """You are OpsPilot — a read-only IT ops copilot for NovaTech.
Answer from structured data only. Never fabricate. State when data is missing.
Structured data: {structured_context}"""

# Structured context builder (same as Phase 3)
def build_structured_context(query):
    q   = query.lower()
    now = datetime.now()
    svc_match = next((s for s in services['service_name'] if s.lower() in q), None)

    m = re.search(r'inc-(\d{4})', q)
    if m:
        row = incidents[incidents['incident_id'] == f'INC-{m.group(1)}']
        if row.empty: return f'No incident INC-{m.group(1)} found.'
        r = row.iloc[0]
        return (f'Incident: INC-{m.group(1)} | Service: {r["service"]} | '
                f'Severity: {r["severity"]} | Status: {r["status"]}\n'
                f'Root cause: {r["root_cause"]} | SLA breached: {r["sla_breached"]}')

    if any(w in q for w in ['sla','breach']):
        b = incidents[incidents['sla_breached']=='Yes']
        by_svc = b.groupby('service').size().sort_values(ascending=False).head(5)
        by_sev = b.groupby('severity').size().reindex(['P1','P2','P3','P4'], fill_value=0)
        return (f'SLA breaches: {len(b)} of {len(incidents)} ({len(b)/len(incidents)*100:.1f}%)\n'
                'By severity: ' + ' | '.join(f'{s}:{c}' for s,c in by_sev.items()) +
                '\nTop services: ' + ' | '.join(f'{s}:{c}' for s,c in by_svc.items()))

    if any(w in q for w in ['root cause','cause','why','pattern']):
        sev = next((s for s in ['P1','P2','P3','P4'] if s.lower() in q), None)
        df  = incidents[incidents['severity']==sev] if sev else incidents
        top = df['root_cause'].value_counts().head(5)
        return 'Top root causes:\n' + '\n'.join(f'  {i+1}. {c}: {n}' for i,(c,n) in enumerate(top.items()))

    if any(w in q for w in ['mttr','resolution time','escalation','down for','how long']):
        df = incidents[incidents['mttr_minutes'].notna()].copy()
        if svc_match: df = df[df['service']==svc_match]
        by_sev  = df.groupby('severity')['mttr_minutes'].mean().reindex(['P1','P2','P3','P4'])
        sla_idx = sla_targets.set_index('severity')
        lines   = [f'MTTR{" for "+svc_match if svc_match else " (all)"}:']
        for s,v in by_sev.items():
            if pd.isna(v): lines.append(f'  {s}: no data')
            else:
                t = sla_idx.loc[s,'sla_mttr_minutes']
                lines.append(f'  {s}: {v:.0f}min (target {t}min)')
        return '\n'.join(lines)

    wm = re.search(r'last (\d+) (day|week|month)', q)
    if wm or any(w in q for w in ['how many','count','total','trend','spike',
                                   'unusual','lately','recently','this month']):
        if wm:
            n,unit = int(wm.group(1)),wm.group(2)
            days = n if unit=='day' else n*7 if unit=='week' else n*30
            df, lbl = incidents[incidents['opened_at']>=now-timedelta(days=days)], f'last {n} {unit}(s)'
        elif 'this month' in q:
            df  = incidents[(incidents['opened_at'].dt.year==now.year)&(incidents['opened_at'].dt.month==now.month)]
            lbl = now.strftime('%B %Y')
        elif any(w in q for w in ['lately','recently','unusual','spike','trend']):
            df, lbl = incidents[incidents['opened_at']>=now-timedelta(days=30)], 'last 30 days'
        else:
            df, lbl = incidents, 'all time'
        if svc_match: df = df[df['service']==svc_match]
        by_sev = df.groupby('severity').size().reindex(['P1','P2','P3','P4'], fill_value=0)
        return (f'Incidents{" for "+svc_match if svc_match else ""} ({lbl}): {len(df)}\n' +
                '\n'.join(f'  {s}: {c}' for s,c in by_sev.items()))

    open_c   = incidents[incidents['status'].isin(['Open','In Progress'])].shape[0]
    breach_c = incidents[incidents['sla_breached']=='Yes'].shape[0]
    return (f'System: {len(incidents)} incidents | {open_c} open | {breach_c} SLA breaches')

def respond_rag(query):
    struct_ctx = build_structured_context(query)
    hits       = retrieve(query, top_k=3)
    kb_ctx     = format_kb_context(hits)
    system_msg = RAG_PROMPT.format(structured_context=struct_ctx, kb_context=kb_ctx)
    t0   = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'system','content':system_msg},{'role':'user','content':query}],
        temperature=0.2, max_tokens=600,
    )
    text = resp.choices[0].message.content.strip()
    ms   = round((time.time()-t0)*1000,1)
    logging.info(json.dumps({'query':_strip_pii(query),'response':_strip_pii(text[:300]),
                              'latency_ms':ms,'kb_hits':len(hits),'agent':'rag-v1'}))
    return {'response':text,'hits':hits,'latency_ms':ms,'tokens':resp.usage.total_tokens}

def respond_no_rag(query):
    struct_ctx = build_structured_context(query)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{'role':'system','content':WITHOUT_RAG_PROMPT.format(structured_context=struct_ctx)},
                  {'role':'user','content':query}],
        temperature=0.2, max_tokens=400,
    )
    return resp.choices[0].message.content.strip()

print('✅ RAG prompt and LLM functions ready')

## Cell 8 — With vs Without RAG: Runbook Queries
These queries **require runbook knowledge** — structured data alone cannot answer them.

In [ ]:
RUNBOOK_QUERIES = [
    'What is the escalation path if auth-service has a P1 incident?',
    'What are the known recurring issues with auth-service?',
    'What should I do if payments-api times out due to a third-party gateway?',
]

print('WITH vs WITHOUT RAG — Runbook Knowledge Queries')
print('='*65)
for i, q in enumerate(RUNBOOK_QUERIES, 1):
    print(f'\nQ{i}: {q}')
    print('─'*65)

    no_rag = respond_no_rag(q)
    print('WITHOUT RAG (structured data only):')
    print(no_rag)

    rag = respond_rag(q)
    print('\nWITH RAG (+ knowledge base):')
    print(rag['response'])
    print(f'\n📎 Retrieved ({len(rag["hits"])} chunks):')
    for h in rag['hits']:
        print(f'   [{h["similarity"]:.3f}] {h["filename"] or h["source"]}')
    time.sleep(0.5)

## Cell 9 — With vs Without RAG: Combined Query
This query benefits from **both** structured data AND runbook knowledge.

In [ ]:
COMBINED_QUERY = 'auth-service has been down for 25 minutes — what do I do?'
print(f'Query: {COMBINED_QUERY}')
print('='*65)

no_rag = respond_no_rag(COMBINED_QUERY)
print('WITHOUT RAG:')
print(no_rag)

rag = respond_rag(COMBINED_QUERY)
print('\nWITH RAG:')
print(rag['response'])
print(f'\nRetrieved chunks:')
for h in rag['hits']:
    print(f'  [{h["similarity"]:.3f}] {h["filename"] or h["source"]}: {h["text"][:100]}...')

## Cell 10 — Missing Data Case: When Retrieval Finds Nothing Relevant

In [ ]:
MISSING_DATA_QUERIES = [
    ('Low relevance — unknown service', 'What is the runbook for service-xyz-unknown?'),
    ('Out of domain entirely',          'What is the weather forecast for Mumbai?'),
    ('Partially covered',               'What is the escalation path for inventory-service?'),
]

print('MISSING DATA CASES — How Agent Handles Low/No Retrieval')
print('='*65)
for label, q in MISSING_DATA_QUERIES:
    hits = retrieve(q, top_k=3)
    rag  = respond_rag(q)
    print(f'\n[{label}]')
    print(f'Query: {q}')
    print(f'Top similarity score: {hits[0]["similarity"]:.3f if hits else 0:.3f}')
    print(f'Response:')
    print(rag['response'])
    time.sleep(0.5)

print('\n' + '='*65)
print('EXPECTED BEHAVIOUR:')
print('  - Unknown service  → agent says no runbook found, escalates')
print('  - Out of domain    → agent declines (scope boundary)')
print('  - Partial coverage → agent uses general_ops.txt + acknowledges gap')

## Cell 11 — Retrieval Quality Summary

In [ ]:
EVAL_QUERIES = [
    ('auth escalation',        'What is the escalation path for auth-service?',           'auth-service.txt'),
    ('payments known issues',  'What are known issues with payments-api?',                'payments-api.txt'),
    ('P1 SLA definition',      'What is the SLA target for a P1 incident?',               'sla_definitions'),
    ('general policy',         'What is the data modification policy for NOC analysts?',  'general_ops.txt'),
    ('auth memory leak',       'Was there a memory leak issue in auth-service?',           'auth-service.txt'),
]

print('RETRIEVAL QUALITY — Expected Source vs Retrieved Source')
print(f'{"Query":<45} {"Expected":>20} {"Top Hit":>20} {"Score":>7} {"✓"}')
print('-'*100)

correct = 0
for label, q, expected_src in EVAL_QUERIES:
    hits = retrieve(q, top_k=1, min_similarity=0.0)  # get top hit regardless of threshold
    top  = hits[0] if hits else None
    got_src  = (top['filename'] or top['source']) if top else 'none'
    score    = top['similarity'] if top else 0
    match    = '✅' if expected_src in got_src else '❌'
    if '✅' in match: correct += 1
    print(f'{q[:43]:<45} {expected_src:>20} {got_src:>20} {score:>7.3f} {match}')

print('-'*100)
print(f'Retrieval accuracy: {correct}/{len(EVAL_QUERIES)} = {correct/len(EVAL_QUERIES)*100:.0f}%')
print(f'(Target: ≥ 80% per Phase 1 success criteria)')

## Cell 12 — Phase 4 Summary

| Capability | Without RAG | With RAG |
|-----------|-------------|----------|
| Incident counts & trends | ✅ | ✅ |
| SLA breach analysis | ✅ | ✅ |
| Escalation procedures | ❌ Hallucinates or says unknown | ✅ From runbook |
| Known recurring issues | ❌ No data | ✅ From runbook |
| Remediation steps | ❌ Generic/guessed | ✅ Service-specific |
| Unknown service runbook | ❌ Hallucinate | ✅ States not found |
| Out-of-domain queries | ⚠️ May answer | ✅ Scope refusal |

**Architecture decisions:**
- Chunk size 300 words / overlap 60 — preserves context at boundaries
- Embedding model: `text-embedding-3-small` — fast, cheap, strong at similarity
- Similarity threshold: 0.25 — filters noise without missing relevant chunks
- Two-source prompt — LLM explicitly told which source to prefer for which question type

**Next → Phase 5:** Add tool-calling so the agent can choose which tool to invoke rather than always running the full pipeline.